# MEC 2026 - Online Music Alignment with Matchmaker (Audio)

**Music Alignment Workshop, MEC 2026 — Section 3 (Online Alignment), Part 1 (Audio)**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pymatchmaker/mec2026_alignment_workshop/blob/online-alignment/notebooks/online_alignment_audio.ipynb)

This notebook accompanies the online alignment section of **Music Alignment Uncovered: Representations, Algorithms and Hands-On Tools** at MEC 2026.

In this notebook we align **one audio recording and the corresponding score** using the [Matchmaker](https://github.com/pymatchmaker/matchmaker). 

## 0. Setup

This notebook is designed for Google Colab, but it also works from a local Jupyter environment.

Run the setup cells once at the beginning. If you are running locally and already installed the dependencies, you may skip the installation cells and start from the imports cell.

In Colab, opening a notebook from GitHub does **not** automatically copy the repository files into the runtime, so the asset setup cell downloads the required `assets/simple_mozart*` files from GitHub when they are not already present locally.

Local note: for audio score following, Matchmaker may need FluidSynth and libsndfile available on the system. In Colab, the first setup cell installs them automatically; locally, install them with conda or your system package manager, for example `conda install -c conda-forge fluidsynth libsndfile`.

In [ ]:
# Colab-only system dependencies.
# Local users can install these separately, e.g.:
# conda install -c conda-forge fluidsynth libsndfile

import importlib.util
import shutil
import subprocess

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB and shutil.which("apt-get") is not None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "fluidsynth", "libsndfile1"],
        check=True,
    )
else:
    print("Skipping apt-get setup outside Colab.")

In [ ]:
# Python dependencies.
# pymatchmaker pulls in the main alignment stack; the other packages are used
# directly in this notebook for plotting, data inspection, and audio features.

%pip install -q "setuptools>=80,<81" "numpy>=1.26.3,<2.0" pymatchmaker pandas matplotlib librosa partitura ipython

In [ ]:
import json
import librosa
import matchmaker
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import partitura as pt
from IPython.display import Image, display
from matchmaker import Matchmaker
from matchmaker.features.audio import ChromagramProcessor, CQTSpectralFluxProcessor
from matchmaker.features.midi import onset_pianoroll
from matchmaker.matchmaker import AVAILABLE_METHODS, DEFAULT_METHOD, DEFAULT_PROCESSOR

print("matchmaker", matchmaker.__version__)

In [ ]:
from pathlib import Path
import urllib.request

ASSET_FILES = [
    "simple_mozart_k265_var1.mp3",
    "simple_mozart_k265_var1.mid",
    "simple_mozart_k265_var1.musicxml",
    "simple_mozart_k265_var1.mei",
    "simple_mozart_k265_var1.match",
    "simple_mozart_k265_var1_note_annotations.txt",
    "simple_mozart_first_two_measures.png",
]

RAW_BASE_URL = "https://raw.githubusercontent.com/pymatchmaker/mec2026_alignment_workshop/main/assets"

def find_or_download_assets():
    candidates = [Path("assets"), Path("../assets")]
    for candidate in candidates:
        if all((candidate / name).exists() for name in ASSET_FILES):
            return candidate

    asset_dir = Path("assets")
    asset_dir.mkdir(exist_ok=True)
    for name in ASSET_FILES:
        target = asset_dir / name
        if not target.exists():
            print(f"Downloading {name}")
            urllib.request.urlretrieve(f"{RAW_BASE_URL}/{name}", target)
    return asset_dir

ASSET_DIR = find_or_download_assets()
SCORE_FILE = ASSET_DIR / "simple_mozart_k265_var1.musicxml"
AUDIO_FILE = ASSET_DIR / "simple_mozart_k265_var1.mp3"
MIDI_FILE = ASSET_DIR / "simple_mozart_k265_var1.mid"
NOTE_ANNOTATIONS = ASSET_DIR / "simple_mozart_k265_var1_note_annotations.txt"
PREVIEW_IMAGE = ASSET_DIR / "simple_mozart_first_two_measures.png"

print("Asset directory:", ASSET_DIR.resolve())
for path in [SCORE_FILE, AUDIO_FILE, MIDI_FILE, NOTE_ANNOTATIONS, PREVIEW_IMAGE]:
    print(path.name, "ok" if path.exists() else "missing")

## 1. Online vs. offline alignment

**Offline alignment** receives the complete performance before it computes the alignment. It can revise earlier decisions because it can see the future.

**Online alignment** receives a stream of observations. At each audio frame or MIDI event, it must estimate the current score position using only the past and present.

Technical constraints:

- Causality: no access to future observations.
- Latency: useful systems must react quickly.
- Tempo variation: the performer may slow down, speed up, or hesitate.
- Ambiguity: repeated or harmonically similar passages can look alike locally.
- Noisy input: audio features can be messy; MIDI can contain missing, extra, or wrong notes.

The Matchmaker pipeline follows this structure:

```text
input source -> Stream -> Processor -> OnlineAlignment -> current score position
(audio/MIDI)    (file or live)  (features)   (method)         (beat position)
```

## 2. The example score and score states

Matchmaker reports the current position in **score beats**. For this Mozart example, the score positions come from Partitura's `note_array()` representation, especially the `onset_beat` field.

The first two measures of the example score are shown below. The unique note-onset beats become the main discrete score states used by several online followers.

In [ ]:
score = pt.load_musicxml(str(SCORE_FILE))
score_part = score[0]
note_array = score_part.note_array()
score_positions = np.unique(note_array["onset_beat"])

print(f"Number of notes: {len(note_array)}")
print(f"Number of unique score onset positions: {len(score_positions)}")
print("First 8 score positions in beats (first measure):")
print(score_positions[:8])

display(Image(filename=str(PREVIEW_IMAGE), width=700))

pd.DataFrame(note_array[:10])[["id", "pitch", "onset_beat", "duration_beat"]]

## 3. Matchmaker overview

For this workshop, the main object is `Matchmaker`. We use it in **simulation mode**: the input is a recorded audio or MIDI performance file, but Matchmaker feeds it through the same streaming pipeline used for real-time score following.

```python
from matchmaker import Matchmaker

mm = Matchmaker(
    score_file="path/to/score.musicxml",
    performance_file="path/to/performance.mp3",  # or .mid
    input_type="audio",                          # or "midi"
    method="arzt",                              # optional
)

for current_position in mm.run():
    print(current_position)
```

The yielded value is the estimated score position in beats. After a run, `mm.score_follower.alignment_path` stores a `(2, T)` array:

- row 0: estimated score beat
- row 1: performance time in seconds

We will use the helper functions below so that each practical example returns the same kind of result dictionary.

In [ ]:
print("Available methods:")
for input_type, methods in AVAILABLE_METHODS.items():
    print(f"  {input_type}: {methods}")

print("\nDefaults:")
print("  method:", DEFAULT_METHOD)
print("  processor:", DEFAULT_PROCESSOR)

In [ ]:
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Results will be saved in {RESULTS_DIR.resolve()}")


def stream_audio_features(audio_file, processor, sample_rate, hop_length):
    """Simulate Matchmaker's hop-by-hop AudioStream feature extraction."""
    audio_y, sr = librosa.load(str(audio_file), sr=None, mono=True)
    if sr != sample_rate:
        audio_y = librosa.resample(y=audio_y, orig_sr=sr, target_sr=sample_rate)

    remainder = len(audio_y) % hop_length
    if remainder > 0:
        audio_y = np.concatenate(
            (audio_y, np.zeros(hop_length - remainder, dtype=np.float32))
        )

    cache_size = getattr(processor, "n_fft", 2 * hop_length) - hop_length
    last_chunk = None
    emit_count = 0
    features = []
    times = []

    for start in range(0, len(audio_y), hop_length):
        target_audio = audio_y[start : start + hop_length]
        if last_chunk is None:
            target_audio = np.concatenate(
                (np.zeros(cache_size, dtype=np.float32), target_audio)
            )
        else:
            target_audio = np.concatenate((last_chunk, target_audio))

        perf_time = emit_count * hop_length / float(sample_rate)
        output = processor((target_audio, perf_time))

        # Matchmaker's AudioStream skips queue emission for the first padded block.
        if last_chunk is not None and output is not None:
            feature_block, feature_time = output
            feature_block = np.asarray(feature_block)
            for row_index, row in enumerate(feature_block):
                features.append(row)
                times.append(feature_time + row_index * hop_length / float(sample_rate))
            emit_count += 1

        last_chunk = target_audio[-cache_size:]

    processor.reset()
    return np.asarray(features), np.asarray(times)


## 4. Audio score following

For audio input, Matchmaker converts audio chunks into frame-based features. The default audio processor is `"chroma"`, and the default audio method is `"arzt"`.

Audio methods available in Matchmaker include:

- `"arzt"`: online time warping adapted from Arzt/Brazier and Widmer style score following.
- `"dixon"`: online time warping based on Dixon's method.
- `"outerhmm"`: outer-product HMM score follower.
- `"skf"`: switching Kalman filter with hidden tempo.

We start by inspecting chroma features from the recorded Mozart performance.

In [ ]:
sample_rate = 44100
frame_rate = 30
hop_length = sample_rate // frame_rate

chroma_processor = ChromagramProcessor(sample_rate=sample_rate, hop_length=hop_length)
chroma, chroma_times = stream_audio_features(
    AUDIO_FILE,
    processor=chroma_processor,
    sample_rate=sample_rate,
    hop_length=hop_length,
)

audio_duration = librosa.get_duration(path=str(AUDIO_FILE))

print("Audio duration:", round(audio_duration, 2), "seconds")
print("Streamed chroma shape:", chroma.shape, "= frames x pitch classes")

fig, ax = plt.subplots(figsize=(10, 4))
img = ax.imshow(
    chroma.T,
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    extent=[chroma_times[0], chroma_times[-1], 0, chroma.shape[1] - 1],
)
ax.set_xlabel("Performance time (seconds)")
ax.set_ylabel("Pitch class")
ax.set_title("Audio chroma features from Matchmaker's streaming processor")
fig.colorbar(img, ax=ax, label="Feature value")
plt.show()

### 4.1 Dynamic programming / Arzt method

The dynamic-programming tracker compares each incoming audio feature frame with the reference features synthesized from the score. It updates a local cost surface and keeps a plausible online path through the score.

Because the tracker is online, it does not get to revise the whole path with future information. Local mistakes and small delays are part of the problem.

In [ ]:
print(f"Performance file: {AUDIO_FILE.name}, with input mode: audio")
print(f"Running Matchmaker with score file ({SCORE_FILE.name}) and method 'arzt'...")

audio_arzt_mm = Matchmaker(
    score_file=SCORE_FILE,
    performance_file=AUDIO_FILE,
    input_type="audio",
    method="arzt",
    processor="chroma",
)

audio_arzt_positions = []
for current_position in audio_arzt_mm.run(verbose=False):
    audio_arzt_positions.append(float(current_position))

audio_arzt_path = audio_arzt_mm.score_follower.alignment_path

print("Tracked audio observations:", len(audio_arzt_positions))
print("Audio alignment path shape:", audio_arzt_path.shape)
print("First 10 estimated beat positions:")
print(np.round(audio_arzt_positions[:10], 3))

audio_arzt_eval = audio_arzt_mm.run_evaluation(
    perf_annotations=NOTE_ANNOTATIONS,
    debug=True,
    save_dir=RESULTS_DIR,
    run_name="audio_arzt",
    level="note",
    plot_dist_matrix=False,
)

print("Evaluation result:")
print(json.dumps(audio_arzt_eval, indent=2))
display(Image(filename=str(RESULTS_DIR / "audio_arzt.png"), width=700))

### 4.2 HMM audio input features: CQT + spectral flux

The audio `outerhmm` configuration uses `processor="cqt_spectral_flux"` for the **incoming performance audio**. Compared with chroma, this representation keeps a wider pitch range and adds a spectral-flux component that emphasizes frame-to-frame changes.

Important distinction: this is the observation feature stream, not the score reference representation. For audio `outerhmm`, `Matchmaker.preprocess_score()` passes the score `note_array()` to `AudioOuterProductHMM`, and the HMM builds symbolic chord/harmonic masks from the score states.

For visualization, we compute the spectral flux directly from the CQT magnitude matrix and append it as the last row. This mirrors the idea of the streaming processor while making the feature visible in one static notebook plot.

In [ ]:
outerhmm_sample_rate = 16000
outerhmm_frame_rate = 25
outerhmm_hop_length = outerhmm_sample_rate // outerhmm_frame_rate

cqt_flux_processor = CQTSpectralFluxProcessor(
    sample_rate=outerhmm_sample_rate,
    hop_length=outerhmm_hop_length,
)
cqt_flux, cqt_flux_times = stream_audio_features(
    AUDIO_FILE,
    processor=cqt_flux_processor,
    sample_rate=outerhmm_sample_rate,
    hop_length=outerhmm_hop_length,
)

cqt_mag = cqt_flux[:, :88]
spectral_flux = cqt_flux[:, 88] if cqt_flux.shape[1] > 88 else np.zeros(len(cqt_flux))

# Normalize only for display so the flux row is visible beside CQT bins.
cqt_display = cqt_mag / (np.max(cqt_mag) + 1e-8)
flux_display = spectral_flux / (np.max(spectral_flux) + 1e-8)
combined_display = np.vstack([cqt_display.T, flux_display[None, :]])

print("Streamed CQT + spectral flux shape:", cqt_flux.shape, "= frames x feature bins")
print("CQT bins:", cqt_mag.shape[1])
print("Spectral flux max:", float(np.max(spectral_flux)))
print("Combined display shape:", combined_display.shape, "= feature rows x frames")

fig, ax = plt.subplots(figsize=(10, 5))
img = ax.imshow(
    combined_display,
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    extent=[cqt_flux_times[0], cqt_flux_times[-1], 0, combined_display.shape[0] - 1],
)
ax.axhline(cqt_mag.shape[1] - 0.5, color="white", linewidth=1.0)
ax.set_xlabel("Performance time (seconds)")
ax.set_ylabel("Feature row")
ax.set_title("OuterHMM input features from Matchmaker's streaming processor")
ax.set_yticks([0, 24, 48, 72, 88])
ax.set_yticklabels(["CQT 0", "CQT 24", "CQT 48", "CQT 72", "Flux"])
fig.colorbar(img, ax=ax, label="Normalized feature value")
plt.show()

# The score reference for audio OuterHMM is symbolic, not CQT.
# Build a lightweight OuterHMM instance to inspect its score-state harmonic masks.
reference_probe_mm = Matchmaker(
    score_file=SCORE_FILE,
    performance_file=AUDIO_FILE,
    input_type="audio",
    method="outerhmm",
)
reference_harmonic_mask = reference_probe_mm.score_follower.chord_harmonic_mask
print(
    "Reference harmonic mask shape:",
    reference_harmonic_mask.shape,
    "= score states x piano keys",
)

fig, ax = plt.subplots(figsize=(10, 4))
img = ax.imshow(
    reference_harmonic_mask.T,
    origin="lower",
    aspect="auto",
    interpolation="nearest",
)
ax.set_xlabel("Score state / chord onset index")
ax.set_ylabel("Piano key index (A0-C8)")
ax.set_title("OuterHMM reference representation: score harmonic masks")
fig.colorbar(img, ax=ax, label="Normalized score-state weight")
plt.show()

### 4.3 HMM / outer-product method

The audio OuterHMM treats the current score position as a hidden state. The reference side is symbolic: the score note array is converted into one harmonic mask per score onset. The observation side is audio: each incoming frame is represented as CQT + spectral flux.

During inference, the CQT part of the observation is compared with each score-state harmonic mask. The spectral-flux value is used as an exit boost, making state changes more likely around acoustic changes.

In [ ]:
audio_outerhmm_mm = None
audio_outerhmm_positions = []
audio_outerhmm_path = None
audio_outerhmm_eval = None

try:
    print(f"Performance file: {AUDIO_FILE.name}, with input mode: audio")
    print(f"Running Matchmaker with score file ({SCORE_FILE.name}) and method 'outerhmm'...")

    audio_outerhmm_mm = Matchmaker(
        score_file=SCORE_FILE,
        performance_file=AUDIO_FILE,
        input_type="audio",
        method="outerhmm",
    )

    for current_position in audio_outerhmm_mm.run(verbose=False):
        audio_outerhmm_positions.append(float(current_position))

    audio_outerhmm_path = audio_outerhmm_mm.score_follower.alignment_path

    print("Tracked audio observations:", len(audio_outerhmm_positions))
    print("OuterHMM alignment path shape:", audio_outerhmm_path.shape)
    print("First 10 estimated beat positions:")
    print(np.round(audio_outerhmm_positions[:10], 3))

    audio_outerhmm_eval = audio_outerhmm_mm.run_evaluation(
        perf_annotations=NOTE_ANNOTATIONS,
        debug=True,
        save_dir=RESULTS_DIR,
        run_name="audio_outerhmm",
        level="note",
        plot_dist_matrix=False,
    )

    print("Evaluation result:")
    print(json.dumps(audio_outerhmm_eval, indent=2))
    display(Image(filename=str(RESULTS_DIR / "audio_outerhmm.png"), width=700))


except Exception as exc:
    print("OuterHMM audio demo skipped:")
    print(type(exc).__name__, exc)

### 4.4 Audio evaluation and method comparison

Now that the DP/Arzt and HMM-style runs have each been inspected independently, we can compare their evaluation metrics in one table.

`run_examples.py` uses `mm.run_evaluation(debug=True, save_dir=..., run_name=...)` after the score follower finishes. The cells above follow the same pattern directly: each completed `Matchmaker` run calls `mm.run_evaluation(...)`, saving alignment paths, ground-truth annotation pairs, JSON metrics, and Matchmaker debug plots into `results/`.

In [ ]:
audio_eval_items = [
    ("arzt + chroma", len(audio_arzt_positions), audio_arzt_eval),
]

if audio_outerhmm_eval is not None:
    audio_eval_items.append(
        ("outerhmm + cqt_spectral_flux", len(audio_outerhmm_positions), audio_outerhmm_eval)
    )

audio_eval_rows = []
for method, observations, evaluation in audio_eval_items:
    row = {"method": method, "observations": observations}
    for key, value in evaluation.items():
        if isinstance(value, dict):
            for subkey, subvalue in value.items():
                row[f"{key}.{subkey}"] = subvalue
        else:
            row[key] = value
    audio_eval_rows.append(row)

pd.DataFrame(audio_eval_rows)

## 5. MIDI score following

For MIDI input, the observations are symbolic note events rather than audio frames. This changes the feature representation and the error model.

MIDI methods available in Matchmaker include:

- `"pthmm"`: pitch-based HMM score follower. This is the default for MIDI.
- `"hmm"`: HMM using pitch and inter-onset information.
- `"outerhmm"`: outer-product HMM for symbolic input.
- `"arzt"` and `"dixon"`: event-based online time warping variants.

The default MIDI processor is `"pitch_chord"`, which groups near-simultaneous note events into chord observations.

In [ ]:
performance = pt.load_performance_midi(str(MIDI_FILE))
performance_notes = performance.note_array()

print("Number of performed MIDI notes:", len(performance_notes))
print("Performance note-array fields:", performance_notes.dtype.names)

midi_pianoroll, midi_onsets = onset_pianoroll(
    performance_notes,
    onset_key="onset_sec",
    piano_range=True,
)

fig, ax = plt.subplots(figsize=(10, 4))
img = ax.imshow(midi_pianoroll.T, origin="lower", aspect="auto", interpolation="nearest")
ax.set_xlabel("Chord/event index")
ax.set_ylabel("Piano key index (A0-C8)")
ax.set_title("MIDI onset piano-roll features")
fig.colorbar(img, ax=ax, label="Active pitch")
plt.show()

pd.DataFrame(performance_notes[:10])[["pitch", "onset_sec", "duration_sec"]]

### 5.1 PitchHMM

`pthmm` is the default MIDI method. It uses pitch observations and a probabilistic state model to estimate which score onset is currently active.

In [ ]:
print(f"Performance file: {MIDI_FILE.name}, with input mode: midi")
print(f"Running Matchmaker with score file ({SCORE_FILE.name}) and method 'pthmm'...")

midi_pthmm_mm = Matchmaker(
    score_file=SCORE_FILE,
    performance_file=MIDI_FILE,
    input_type="midi",
    method="pthmm",
)

midi_pthmm_positions = []
for current_position in midi_pthmm_mm.run(verbose=False):
    midi_pthmm_positions.append(float(current_position))

midi_pthmm_path = midi_pthmm_mm.score_follower.alignment_path

print("MIDI observations:", len(midi_pthmm_positions))
print("Alignment path shape:", midi_pthmm_path.shape)
print("First 10 estimated beat positions:")
print(np.round(midi_pthmm_positions[:10], 3))

midi_pthmm_eval = midi_pthmm_mm.run_evaluation(
    perf_annotations=NOTE_ANNOTATIONS,
    debug=True,
    save_dir=RESULTS_DIR,
    run_name="midi_pthmm",
    level="note",
    plot_dist_matrix=False,
)

print("Evaluation result:")
print(json.dumps(midi_pthmm_eval, indent=2))
display(Image(filename=str(RESULTS_DIR / "midi_pthmm.png"), width=700))

### 5.2 OuterHMM

Now run the symbolic OuterHMM separately. Keeping this plot separate from PitchHMM makes it easier to inspect the path shape and any failures.

In [ ]:
midi_outerhmm_mm = None
midi_outerhmm_positions = []
midi_outerhmm_path = None
midi_outerhmm_eval = None

try:
    print(f"Performance file: {MIDI_FILE.name}, with input mode: midi")
    print(f"Running Matchmaker with score file ({SCORE_FILE.name}) and method 'outerhmm'...")

    midi_outerhmm_mm = Matchmaker(
        score_file=SCORE_FILE,
        performance_file=MIDI_FILE,
        input_type="midi",
        method="outerhmm",
    )

    for current_position in midi_outerhmm_mm.run(verbose=False):
        midi_outerhmm_positions.append(float(current_position))

    midi_outerhmm_path = midi_outerhmm_mm.score_follower.alignment_path

    print("MIDI OuterHMM observations:", len(midi_outerhmm_positions))
    print("Alignment path shape:", midi_outerhmm_path.shape)
    print("First 10 estimated beat positions:")
    print(np.round(midi_outerhmm_positions[:10], 3))

    midi_outerhmm_eval = midi_outerhmm_mm.run_evaluation(
        perf_annotations=NOTE_ANNOTATIONS,
        debug=True,
        save_dir=RESULTS_DIR,
        run_name="midi_outerhmm",
        level="note",
        plot_dist_matrix=False,
    )

    print("Evaluation result:")
    print(json.dumps(midi_outerhmm_eval, indent=2))
    display(Image(filename=str(RESULTS_DIR / "midi_outerhmm.png"), width=700))
except Exception as exc:
    print("MIDI OuterHMM demo skipped:")
    print(type(exc).__name__, exc)

## 6. MIDI evaluation

Compare the MIDI PitchHMM and MIDI OuterHMM evaluation metrics in one table.

In [ ]:
midi_eval_items = [
    ("midi pthmm", len(midi_pthmm_positions), midi_pthmm_eval),
]

if midi_outerhmm_eval is not None:
    midi_eval_items.append(
        ("midi outerhmm", len(midi_outerhmm_positions), midi_outerhmm_eval)
    )

midi_eval_rows = []
for method, observations, evaluation in midi_eval_items:
    row = {"method": method, "observations": observations}
    for key, value in evaluation.items():
        if isinstance(value, dict):
            for subkey, subvalue in value.items():
                row[f"{key}.{subkey}"] = subvalue
        else:
            row[key] = value
    midi_eval_rows.append(row)

pd.DataFrame(midi_eval_rows)

## 7. Practical variations

Small changes to try during the workshop:

Audio:

- Change the audio method from `"arzt"` to `"dixon"`.
- Change the audio processor from `"chroma"` to `"mfcc"`, `"cqt"`, or `"mel"`.
- Inspect the saved files in `results/`: alignment path TSV, ground-truth TSV, JSON metrics, and debug plot.

MIDI:

- Try MIDI `"hmm"` or `"arzt"` in the playground cell and inspect each plot separately.
- Look for places where symbolic methods react differently to event grouping or pitch errors.

The important thing to watch is not only whether a final alignment looks correct, but how quickly and stably the online estimate moves while the performance unfolds.

In [ ]:
# Example playground cell.
# Edit these values and rerun.

PLAY_INPUT_TYPE = "audio"   # "audio" or "midi"
PLAY_METHOD = "dixon"       # audio: "arzt", "dixon", "outerhmm"; midi: "pthmm", "hmm", "outerhmm", "arzt"
PLAY_PROCESSOR = None        # e.g. "chroma", "mfcc", "cqt", "mel" for audio

performance_file = AUDIO_FILE if PLAY_INPUT_TYPE == "audio" else MIDI_FILE
run_name = f"playground_{PLAY_INPUT_TYPE}_{PLAY_METHOD}"

try:
    playground_mm = Matchmaker(
        score_file=SCORE_FILE,
        performance_file=performance_file,
        input_type=PLAY_INPUT_TYPE,
        method=PLAY_METHOD,
        processor=PLAY_PROCESSOR,
    )
    playground_positions = []
    for current_position in playground_mm.run(verbose=False):
        playground_positions.append(float(current_position))

    playground_eval = playground_mm.run_evaluation(
        perf_annotations=NOTE_ANNOTATIONS,
        debug=True,
        save_dir=RESULTS_DIR,
        run_name=run_name,
        level="note",
        plot_dist_matrix=False,
    )

    print("Tracked observations:", len(playground_positions))
    print("Evaluation result:")
    print(json.dumps(playground_eval, indent=2))
    display(Image(filename=str(RESULTS_DIR / f"{run_name}.png"), width=700))
except Exception as exc:
    print("Playground run failed:")
    print(type(exc).__name__, exc)

## 8. Optional extension: where custom score followers plug in

This is not the main workshop path. The practical priority is to run and compare the existing Matchmaker followers above.

If you later want to add a new online alignment method, the core interface is small: subclass `matchmaker.base.OnlineAlignment`, implement `step(features)`, and update `self.current_index`. Matchmaker's base class handles `__call__`, `run()`, and `alignment_path` bookkeeping.

The tiny example below is intentionally naive: it ignores the input features and simply advances one score state per observation. Its purpose is only to show the interface described in `HOW_TO_MAKE_CUSTOM_SCORE_FOLLOWERS.md`.

In [ ]:
from matchmaker.base import OnlineAlignment

class StepThroughFollower(OnlineAlignment):
    def step(self, features):
        self.current_index = min(
            self.current_index + 1,
            len(self.score_positions) - 1,
        )

toy_follower = StepThroughFollower(score_positions=score_positions)

for i in range(8):
    beat = toy_follower((None, i * 0.25))
    print(f"observation {i:02d}: estimated score beat = {beat:.2f}")

print("Toy alignment path shape:", toy_follower.alignment_path.shape)